# 🏀 Basketball Player Detection & Tracking System
## Big Vision Internship Assignment — Complete Implementation

---

## 📌 PROJECT OVERVIEW

This notebook builds a **complete end-to-end pipeline** for detecting and tracking basketball players in game videos.

| Component | Choice | Reason |
|-----------|--------|--------|
| **Detection Model** | YOLOv11m | 22% fewer parameters than YOLOv8 with higher mAP. Real-time on T4 GPU. Single-stage = fast. |
| **Tracking Algorithm** | ByteTrack | Best MOTA (77.3%), fewest ID switches, 171 FPS. Handles occlusions best. |
| **Dataset** | 3 Roboflow datasets merged | ~3600+ diverse labeled images for maximum accuracy |

---

## 🧠 WHY BYTETRACK? (Not SORT, Not DeepSORT)

| Algorithm | How it works | Problem for Basketball |
|-----------|-------------|----------------------|
| **SORT** | Kalman filter only | Players cross paths → loses track → ID #3 becomes #8 ❌ |
| **DeepSORT** | Motion + appearance (CNN) | All players wear similar jerseys → appearance model confused ❌ |
| **ByteTrack ✅** | Uses ALL detections (high + low confidence) | When player is half-hidden, low-confidence box keeps them tracked ✅ |

ByteTrack's key insight: instead of throwing away low-confidence detections (like SORT/DeepSORT do), it uses them in a second matching pass to recover tracks during occlusion. This is exactly what basketball needs.

---

## ⚠️ BEFORE YOU RUN — DO THESE 3 THINGS FIRST

### 1️⃣ Enable GPU
`Runtime` → `Change runtime type` → **T4 GPU** → Save

### 2️⃣ Get Free Roboflow API Key
1. Go to **https://roboflow.com** → Sign Up (free)
2. Profile icon → Settings → Roboflow API → Copy key
3. Paste it in **Cell 2** where it says `YOUR_API_KEY_HERE`

### 3️⃣ Run Cells Top to Bottom — Never Skip Any Cell

---

## 📂 WHAT THIS NOTEBOOK COVERS
1. Install libraries
2. Download 3 basketball datasets via Roboflow API
3. Quality filter (remove blurry/dark/corrupted images)
4. Preprocess images (resize to 640×640)
5. Merge datasets with unified class IDs
6. Verify annotations + visualize samples
7. Train YOLOv11m (fine-tuning, ~90 min on T4)
8. Evaluate detection (mAP, Precision, Recall, F1)
9. Download test video
10. Extract + preprocess frames from video
11. Run full tracking with ByteTrack
12. Compute tracking statistics
13. Generate player movement heatmap
14. Save everything to Google Drive
15. Gradio live demo (drag-and-drop video input)


---
## 📦 CELL 1 — Install All Required Libraries

**What this does:**
Installs every Python library needed for the project. Run this once per Colab session.

| Library | Purpose |
|---------|---------|
| `ultralytics` | YOLOv11 model — detection + tracking |
| `roboflow` | Download labeled datasets |
| `supervision` | Video annotation, box drawing, trails |
| `lap` | Linear Assignment Problem solver (needed by ByteTrack) |
| `gradio` | Live demo UI at the end |

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 1 — INSTALL ALL REQUIRED LIBRARIES
# ================================================================
# What: Installs ultralytics (YOLOv11), roboflow (datasets),
#       supervision (video annotation), lap (ByteTrack dependency),
#       gradio (live demo UI at the end)
# Why:  These are all the tools needed for detection + tracking
# Change: Nothing. Just run.
# ================================================================

!pip install ultralytics==8.3.40 -q
!pip install roboflow -q
!pip install supervision -q
!pip install lap -q
!pip install gradio -q

# Standard libraries (already in Colab)
import os, cv2, shutil, glob, yaml, random, warnings, zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image as PILImage
import torch
import pandas as pd

warnings.filterwarnings('ignore')

# ── Verify GPU is active ───────────────────────────────────────
print('=' * 60)
print('  LIBRARY CHECK')
print('=' * 60)
print(f'  PyTorch version : {torch.__version__}')
print(f'  GPU Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU Name        : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU Memory      : {mem:.1f} GB')
    print('  Status          : READY ✅')
else:
    print()
    print('  ❌ NO GPU DETECTED!')
    print('  Fix: Runtime > Change runtime type > T4 GPU > Save')
    print('  Then: Runtime > Run all')
print('=' * 60)
print('  All libraries installed successfully!')
print('=' * 60)


---
## 📥 CELL 2 — Download All 3 Datasets via Roboflow API

**What this does:**
Downloads 3 labeled basketball datasets directly into Colab using the Roboflow API.
All datasets are in **YOLOv8 format** (compatible with YOLOv11).

| Dataset | Images | Content |
|---------|--------|---------|
| Dataset 1 | ~1,616 | Basketball players, labeled bounding boxes |
| Dataset 2 | ~1,398 | More basketball players, different angles |
| Dataset 3 | ~600+ | Roboflow official basketball dataset |

**Why 3 datasets?**
More diverse training data = model learns different court types, lighting, camera angles = higher accuracy.

**⚠️ YOU MUST CHANGE ONE THING:**
Replace `YOUR_API_KEY_HERE` with your actual Roboflow API key.
Get it free: **https://roboflow.com → Settings → Roboflow API**

In [ ]:
# ================================================================
# CELL 2 — DOWNLOAD DATASETS VIA ROBOFLOW API
# ================================================================
# What: Downloads 3 basketball datasets (total ~3600+ images)
# Why:  More diverse data = higher detection accuracy
# Where datasets go: /content/dataset1, /content/dataset2, /content/dataset3
#
# ⚠️  CHANGE THIS: Replace YOUR_API_KEY_HERE with your Roboflow key
# Get free key: roboflow.com → Settings → Roboflow API
# ================================================================

from roboflow import Roboflow

# ══════════════════════════════════════════════════
# ⚠️  ONLY CHANGE THIS ONE LINE — your Roboflow API key
API_KEY = 'YOUR_API_KEY_HERE'
# ══════════════════════════════════════════════════

rf = Roboflow(api_key=API_KEY)

# --- Dataset 1: Basketball Player Detection (1616 images) ---
print('📦 Downloading Dataset 1 (~1616 images)...')
print('   Source: lokesh-podipireddy-eocdt / basketball-player-detection')
try:
    p1 = rf.workspace('lokesh-podipireddy-eocdt').project('basketball-player-detection-6y9yj')
    d1 = p1.version(1).download('yolov8', location='/content/dataset1')
    print('   ✅ Dataset 1 ready at: /content/dataset1')
except Exception as e:
    print(f'   ❌ Dataset 1 error: {e}')

# --- Dataset 2: Basketball Player Detection 2 (1398 images) ---
print()
print('📦 Downloading Dataset 2 (~1398 images)...')
print('   Source: roboflow-jvuqo / basketball-player-detection-2')
try:
    p2 = rf.workspace('roboflow-jvuqo').project('basketball-player-detection-2')
    d2 = p2.version(1).download('yolov8', location='/content/dataset2')
    print('   ✅ Dataset 2 ready at: /content/dataset2')
except Exception as e:
    print(f'   ❌ Dataset 2 error: {e}')

# --- Dataset 3: Roboflow Official Basketball (600+ images) ---
print()
print('📦 Downloading Dataset 3 (~600+ images)...')
print('   Source: roboflow-universe-projects / basketball-players-fy4c2')
try:
    p3 = rf.workspace('roboflow-universe-projects').project('basketball-players-fy4c2')
    d3 = p3.version(4).download('yolov8', location='/content/dataset3')
    print('   ✅ Dataset 3 ready at: /content/dataset3')
except Exception as e:
    print(f'   ❌ Dataset 3 error: {e}')

# --- Final count ---
print()
print('📊 Download Summary:')
total_images = 0
for i, path in enumerate(['/content/dataset1', '/content/dataset2', '/content/dataset3'], 1):
    if os.path.exists(path):
        n = len(glob.glob(f'{path}/**/*.jpg', recursive=True) +
                glob.glob(f'{path}/**/*.png', recursive=True))
        print(f'   Dataset {i}: {n} images found')
        total_images += n
    else:
        print(f'   Dataset {i}: NOT FOUND (download may have failed)')
print(f'   TOTAL   : {total_images} images')
print()
print('✅ Dataset download complete!')


---
## 🔍 CELL 3 — Smart Dataset Path Detection (Fixes "0 Images Found" Error)

**What this does:**
Different Roboflow datasets organize their folders in different ways:
- Some use `dataset/images/train/` 
- Some use `dataset/train/images/`

This cell automatically detects **which layout each dataset uses** and normalizes everything to a consistent structure before merging.

**This is the fix for the "0 images found" error.**

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 3 — SMART PATH DETECTION
# ================================================================
# What: Roboflow datasets use 2 different folder layouts.
#       This fixes the "0 images found" error by auto-detecting
#       which layout each dataset uses and mapping it correctly.
#
# Layout A (standard): dataset/images/train/  + dataset/labels/train/
# Layout B (alternate): dataset/train/images/ + dataset/train/labels/
#
# This function handles BOTH layouts automatically.
# Why: Without this, merging fails silently with 0 images copied.
# Change: Nothing. Just run.
# ================================================================

def get_yolo_paths(dataset_root, split):
    """
    Smart path finder that handles both Roboflow folder layouts.
    Returns (images_dir, labels_dir) or (None, None) if not found.

    Layout A: dataset_root/images/train/ + dataset_root/labels/train/
    Layout B: dataset_root/train/images/ + dataset_root/train/labels/
    Also handles 'val' vs 'valid' naming.
    """
    split_aliases = [split]
    if split == 'valid':
        split_aliases.append('val')
    elif split == 'val':
        split_aliases.append('valid')

    for s in split_aliases:
        # Layout A: images/split/
        img_a = os.path.join(dataset_root, 'images', s)
        lbl_a = os.path.join(dataset_root, 'labels', s)
        if os.path.exists(img_a):
            imgs = (glob.glob(f'{img_a}/*.jpg') +
                    glob.glob(f'{img_a}/*.png') +
                    glob.glob(f'{img_a}/*.jpeg'))
            if imgs:
                return img_a, lbl_a

        # Layout B: split/images/
        img_b = os.path.join(dataset_root, s, 'images')
        lbl_b = os.path.join(dataset_root, s, 'labels')
        if os.path.exists(img_b):
            imgs = (glob.glob(f'{img_b}/*.jpg') +
                    glob.glob(f'{img_b}/*.png') +
                    glob.glob(f'{img_b}/*.jpeg'))
            if imgs:
                return img_b, lbl_b

    return None, None


def inspect_dataset(path, name):
    """Show what's inside a downloaded dataset."""
    print(f'\n--- {name} ---')
    print(f'    Path: {path}')
    if not os.path.exists(path):
        print('    ❌ PATH NOT FOUND')
        return

    yaml_path = os.path.join(path, 'data.yaml')
    if os.path.exists(yaml_path):
        with open(yaml_path) as f:
            info = yaml.safe_load(f)
        print(f'    Classes ({info.get("nc","?")}): {info.get("names", [])}')

    total = 0
    for split in ['train', 'valid', 'test']:
        img_dir, _ = get_yolo_paths(path, split)
        if img_dir:
            imgs = (glob.glob(f'{img_dir}/*.jpg') +
                    glob.glob(f'{img_dir}/*.png') +
                    glob.glob(f'{img_dir}/*.jpeg'))
            n = len(imgs)
            print(f'    {split:6s}: {n:4d} images  (found at: {img_dir})')
            total += n
    print(f'    TOTAL : {total} images')


print('🔍 Inspecting all downloaded datasets...')
inspect_dataset('/content/dataset1', 'Dataset 1 — Lokesh Basketball')
inspect_dataset('/content/dataset2', 'Dataset 2 — Roboflow Basketball 2')
inspect_dataset('/content/dataset3', 'Dataset 3 — Roboflow Official')
print()
print('✅ Path detection complete! Smart helper is ready.')


---
## 🧹 CELL 4 — Quality Filter (Remove Bad Training Images)

**What this does:**
Scans every image across all 3 datasets and removes:
- **Blurry frames** — Laplacian variance < 80 (motion blur from fast players)
- **Too dark images** — average brightness < 25 (underexposed footage)
- **Overexposed images** — average brightness > 235 (blown out court lighting)
- **Corrupted files** — images that can't be opened

When an image is removed, its matching label file is removed too.

**Why this matters:**
Training on blurry/dark images teaches the model bad habits. Removing 5% bad images can improve mAP by several points.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 4 — QUALITY FILTER
# ================================================================
# What: Removes blurry, too-dark, overexposed, and corrupted images
#       from all 3 datasets before merging.
# Why:  Bad training images reduce model accuracy.
#       Better 3000 clean images than 3500 noisy ones.
# How:
#   - Blur check: Laplacian variance (low = blurry)
#   - Brightness: HSV Value channel mean
#   - Corruption: PIL verify()
# Change: Nothing. Just run.
# ================================================================

def quality_filter(dataset_path, blur_thresh=80, min_bright=25, max_bright=235):
    """
    Removes low-quality images and their paired label files.
    Returns (kept_count, removed_count).
    """
    removed = 0
    kept    = 0

    for split in ['train', 'valid', 'val', 'test']:
        img_dir, lbl_dir = get_yolo_paths(dataset_path, split)
        if not img_dir:
            continue

        images = (glob.glob(f'{img_dir}/*.jpg') +
                  glob.glob(f'{img_dir}/*.png') +
                  glob.glob(f'{img_dir}/*.jpeg'))

        for img_path in images:
            reason = None

            # 1. Check for file corruption
            try:
                PILImage.open(img_path).verify()
            except Exception:
                reason = 'corrupted'

            # 2. Check for blur (Laplacian variance method)
            if reason is None:
                gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if gray is None:
                    reason = 'unreadable'
                else:
                    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
                    if blur_score < blur_thresh:
                        reason = f'blurry(score={blur_score:.0f})'

            # 3. Check brightness
            if reason is None:
                bgr = cv2.imread(img_path)
                if bgr is not None:
                    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
                    brightness = hsv[:, :, 2].mean()
                    if brightness < min_bright:
                        reason = f'too_dark(brightness={brightness:.0f})'
                    elif brightness > max_bright:
                        reason = f'too_bright(brightness={brightness:.0f})'

            if reason:
                # Remove image
                os.remove(img_path)
                # Remove matching label file
                if lbl_dir:
                    lbl_path = os.path.join(lbl_dir, Path(img_path).stem + '.txt')
                    if os.path.exists(lbl_path):
                        os.remove(lbl_path)
                removed += 1
            else:
                kept += 1

    return kept, removed


print('🧹 Running quality filter on all 3 datasets...')
print('   Thresholds: blur>80, brightness 25–235')
print()

for i, path in enumerate(['/content/dataset1',
                           '/content/dataset2',
                           '/content/dataset3'], 1):
    if os.path.exists(path):
        k, r = quality_filter(path)
        status = '✅' if r < k * 0.1 else '⚠️'
        print(f'   {status} Dataset {i}: kept={k}, removed={r} ({r/(k+r)*100:.1f}% removed)')
    else:
        print(f'   ❌ Dataset {i}: not found, skipping')

print()
print('✅ Quality filter complete!')


---
## ⚙️ CELL 5 — Preprocess Images (Resize + Normalize)

**What this does:**
- Resizes all images to **640×640 pixels** (standard YOLO input size)
- Converts any grayscale images to **RGB (3 channels)**
- Saves with **95% JPEG quality** (sharp enough, not too heavy)

**Why 640×640?**
YOLOv11 processes square images internally. If you feed different sizes, it resizes anyway — but doing it beforehand ensures consistent quality and faster data loading during training.

**Why doesn't resizing break the labels?**
YOLO labels use **normalized coordinates (0.0 to 1.0)**, not pixel values. So `cx=0.5` means center of image regardless of whether it's 640 or 1920 pixels wide.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 5 — PREPROCESS IMAGES
# ================================================================
# What: Resize all images to 640x640, convert grayscale to RGB
# Why:  YOLOv11 needs 640x640 input. Consistent size = faster training.
#       YOLO labels use normalized 0-1 coords so resizing does NOT
#       break the bounding box annotations.
# Change: Nothing. Just run.
# ================================================================

def preprocess_images(dataset_path):
    """
    Resize all images to 640x640 and convert grayscale to RGB.
    Does NOT modify label files (YOLO labels are normalized 0-1).
    """
    resized   = 0
    converted = 0
    errors    = 0

    for split in ['train', 'valid', 'val', 'test']:
        img_dir, _ = get_yolo_paths(dataset_path, split)
        if not img_dir:
            continue

        images = (glob.glob(f'{img_dir}/*.jpg') +
                  glob.glob(f'{img_dir}/*.png') +
                  glob.glob(f'{img_dir}/*.jpeg'))

        for img_path in images:
            try:
                img = cv2.imread(img_path)
                if img is None:
                    errors += 1
                    continue

                # Convert grayscale (2D) to BGR (3 channel)
                if len(img.shape) == 2:
                    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
                    converted += 1

                # Resize to 640x640
                h, w = img.shape[:2]
                if (w, h) != (640, 640):
                    img = cv2.resize(img, (640, 640),
                                     interpolation=cv2.INTER_LINEAR)
                    resized += 1

                # Save with high quality
                cv2.imwrite(img_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])

            except Exception:
                errors += 1

    return resized, converted, errors


print('⚙️  Preprocessing all datasets to 640×640...')
print()

for i, path in enumerate(['/content/dataset1',
                           '/content/dataset2',
                           '/content/dataset3'], 1):
    if os.path.exists(path):
        r, c, e = preprocess_images(path)
        print(f'   Dataset {i}: resized={r}, converted_gray={c}, errors={e}')
    else:
        print(f'   Dataset {i}: not found, skipping')

print()
print('✅ Preprocessing complete! All images are 640×640 RGB.')


---
## 🔀 CELL 6 — Merge All 3 Datasets Into One

**What this does:**
Combines all 3 datasets into a single `/content/merged_dataset` folder with:
- Unified class IDs: `player=0`, `referee=1`, `ball=2`
- Proper train/valid/test split structure
- Renamed files with dataset prefix (avoids filename conflicts)
- A unified `data.yaml` config file for training

**Why merge?**
Each dataset has images from different cameras, angles, and lighting. Merging gives the model more diverse training data = significantly higher accuracy.

**Class ID remapping:**
Different datasets may label "player" as class 0 or class 1. This cell auto-detects and fixes all class IDs so they all use the same unified mapping.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 6 — MERGE ALL 3 DATASETS INTO ONE
# ================================================================
# What: Combines dataset1 + dataset2 + dataset3 into merged_dataset
# Why:  3600+ diverse images >> 1600 single-source images for accuracy
# How:
#   - Uses get_yolo_paths() to handle both folder layouts
#   - Remaps class IDs to unified: player=0, referee=1, ball=2
#   - Prefixes filenames with ds0_, ds1_, ds2_ to avoid conflicts
#   - Creates data.yaml config for training
# Change: Nothing. Just run.
# ================================================================

TARGET_CLASSES = {0: 'player', 1: 'referee', 2: 'ball'}

def get_class_mapping(yaml_path):
    """Read class names from a dataset's data.yaml file."""
    if not os.path.exists(yaml_path):
        return {0: 'player'}
    with open(yaml_path) as f:
        info = yaml.safe_load(f)
    return {i: n.lower() for i, n in enumerate(info.get('names', ['player']))}

def remap_class_id(orig_id, src_map):
    """
    Map any dataset's class ID to our unified class ID.
    player/person/athlete → 0
    referee/ref/official  → 1
    ball/basketball       → 2
    """
    name = src_map.get(orig_id, 'player').lower()
    if any(k in name for k in ['player', 'person', 'athlete', 'human']):
        return 0
    elif any(k in name for k in ['referee', 'ref', 'official']):
        return 1
    elif any(k in name for k in ['ball', 'basketball']):
        return 2
    return 0  # default to player

def merge_datasets(src_dirs, target_dir):
    """
    Merge multiple YOLOv8 datasets into one unified dataset.
    Handles both folder layouts using get_yolo_paths().
    """
    os.makedirs(target_dir, exist_ok=True)
    for split in ['train', 'valid', 'test']:
        os.makedirs(f'{target_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{target_dir}/labels/{split}', exist_ok=True)

    counts = {'train': 0, 'valid': 0, 'test': 0}

    for ds_idx, src in enumerate(src_dirs):
        if not os.path.exists(src):
            print(f'   ⚠️  Skipping missing dataset: {src}')
            continue

        src_map = get_class_mapping(os.path.join(src, 'data.yaml'))
        print(f'   Dataset {ds_idx+1} class map: {src_map}')

        for tgt_split in ['train', 'valid', 'test']:
            img_dir, lbl_dir = get_yolo_paths(src, tgt_split)
            if not img_dir:
                continue

            images = (glob.glob(f'{img_dir}/*.jpg') +
                      glob.glob(f'{img_dir}/*.png') +
                      glob.glob(f'{img_dir}/*.jpeg'))

            for img_path in images:
                stem     = Path(img_path).stem
                ext      = Path(img_path).suffix
                new_name = f'ds{ds_idx}_{stem}'

                # Copy image
                dst_img = f'{target_dir}/images/{tgt_split}/{new_name}{ext}'
                shutil.copy(img_path, dst_img)

                # Copy + remap label
                src_lbl = os.path.join(lbl_dir, stem + '.txt') if lbl_dir else ''
                dst_lbl = f'{target_dir}/labels/{tgt_split}/{new_name}.txt'

                if src_lbl and os.path.exists(src_lbl):
                    new_lines = []
                    with open(src_lbl) as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) == 5:
                                new_cls = remap_class_id(int(parts[0]), src_map)
                                new_lines.append(
                                    f'{new_cls} {" ".join(parts[1:])}\n')
                    with open(dst_lbl, 'w') as f:
                        f.writelines(new_lines)
                else:
                    open(dst_lbl, 'w').close()  # empty = background image

                counts[tgt_split] += 1

    # Create unified data.yaml
    data_yaml = {
        'path'  : target_dir,
        'train' : 'images/train',
        'val'   : 'images/valid',
        'test'  : 'images/test',
        'nc'    : 3,
        'names' : ['player', 'referee', 'ball']
    }
    with open(f'{target_dir}/data.yaml', 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)

    total = sum(counts.values())
    print(f'\n   Merge complete!')
    for split, count in counts.items():
        print(f'     {split:6s}: {count:4d} images')
    print(f'     TOTAL : {total} images')
    print(f'     Config: {target_dir}/data.yaml')
    return total


print('🔀 Merging all 3 datasets...')
print()
total_merged = merge_datasets(
    src_dirs   = ['/content/dataset1',
                  '/content/dataset2',
                  '/content/dataset3'],
    target_dir = '/content/merged_dataset'
)
print()
print(f'✅ Merged dataset ready at: /content/merged_dataset')
print(f'   Total images: {total_merged}')


---
## ✅ CELL 7 — Verify Annotations + Class Distribution Chart

**What this does:**
1. **Annotation Verification** — checks every `.txt` label file has valid YOLO format
2. **Class Distribution Chart** — bar chart showing how many player/referee/ball annotations exist in each split

**YOLO label format (what we verify):**
Each line = one object: `class_id  cx  cy  width  height`
All values normalized 0.0–1.0. Example: `0 0.512 0.438 0.089 0.201`

**Why verify?**
A single malformed label file can crash training. Better to catch errors now.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 7 — VERIFY ANNOTATIONS + CLASS DISTRIBUTION CHART
# ================================================================
# What: (1) Checks every label file is valid YOLO format
#       (2) Plots class distribution across train/valid/test splits
# Why:  Malformed labels crash training. Class imbalance hurts accuracy.
#       If 'ball' has 10x fewer samples, model won't detect it well.
# Change: Nothing. Just run.
# ================================================================

# ── Part 1: Annotation Verification ──────────────────────────
print('🔍 Verifying all annotation files...')
error_list = []
total_files = 0
valid_files = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = f'/content/merged_dataset/labels/{split}'
    if not os.path.exists(lbl_dir):
        continue
    for lbl_path in glob.glob(f'{lbl_dir}/*.txt'):
        total_files += 1
        file_ok = True
        with open(lbl_path) as f:
            for line_num, line in enumerate(f):
                parts = line.strip().split()
                if not parts:
                    continue  # empty line is fine
                if len(parts) != 5:
                    error_list.append(f'Bad format ({len(parts)} values): {Path(lbl_path).name}:{line_num}')
                    file_ok = False
                    continue
                try:
                    cls_id = int(parts[0])
                    vals   = list(map(float, parts[1:]))
                    if not all(0.0 <= v <= 1.0 for v in vals):
                        error_list.append(f'Out of range: {Path(lbl_path).name}:{line_num}')
                        file_ok = False
                    if cls_id not in [0, 1, 2]:
                        error_list.append(f'Unknown class {cls_id}: {Path(lbl_path).name}:{line_num}')
                        file_ok = False
                except ValueError:
                    error_list.append(f'Non-numeric: {Path(lbl_path).name}:{line_num}')
                    file_ok = False
        if file_ok:
            valid_files += 1

print(f'   Total label files : {total_files}')
print(f'   Valid files       : {valid_files}')
print(f'   Errors found      : {len(error_list)}')
if error_list[:5]:
    for e in error_list[:5]:
        print(f'   ⚠️  {e}')
print()

# ── Part 2: Class Distribution Chart ─────────────────────────
print('📊 Generating class distribution chart...')
counts = {s: {0: 0, 1: 0, 2: 0} for s in ['train', 'valid', 'test']}
for split in ['train', 'valid', 'test']:
    lbl_dir = f'/content/merged_dataset/labels/{split}'
    if not os.path.exists(lbl_dir):
        continue
    for lbl_path in glob.glob(f'{lbl_dir}/*.txt'):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    c = int(parts[0])
                    if c in counts[split]:
                        counts[split][c] += 1

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2196F3', '#F44336', '#FF9800']
names  = ['player', 'referee', 'ball']

for ax, split in zip(axes, ['train', 'valid', 'test']):
    vals = [counts[split][k] for k in [0, 1, 2]]
    bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 5,
                    f'{int(h):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(f'{split.capitalize()} Split\nTotal annotations: {sum(vals):,}', fontsize=11)
    ax.set_ylabel('Annotation Count')
    ax.set_ylim(0, max(max(vals)*1.15, 10))

fig.suptitle('Class Distribution — Merged Basketball Dataset\n'
             'Blue=Player  Red=Referee  Orange=Ball', fontsize=13)
plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print('✅ Chart saved: /content/class_distribution.png')
print(f'✅ Annotations valid: {len(error_list)==0}')


---
## 🖼️ CELL 8 — Visualize Annotated Training Samples

**What this does:**
Shows 9 random training images with their bounding box annotations drawn on top.

**Color coding:**
- 🔵 **Blue** = Player
- 🔴 **Red** = Referee
- 🟠 **Orange** = Ball

**Why look at this?**
Before spending 90 minutes training, visually confirm the annotations look correct. If bounding boxes are wrong or classes are swapped, you'd know now instead of after training.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 8 — VISUALIZE ANNOTATED TRAINING SAMPLES
# ================================================================
# What: Shows 9 random training images with bounding boxes drawn
# Why:  Visual sanity check before training.
#       Confirms annotations are correct (right class, right box).
# Color: Blue=Player, Red=Referee, Orange=Ball
# Change: Nothing. Just run.
# ================================================================

images_list = (glob.glob('/content/merged_dataset/images/train/*.jpg') +
               glob.glob('/content/merged_dataset/images/train/*.png'))

if not images_list:
    print('❌ No images found in merged_dataset/images/train/')
    print('   Make sure Cell 6 (merge) ran successfully.')
else:
    random.shuffle(images_list)
    sample_imgs = images_list[:9]

    CLASS_COLORS = {0: (0.2, 0.6, 1.0),    # Blue   = player
                    1: (1.0, 0.25, 0.25),   # Red    = referee
                    2: (1.0, 0.6,  0.0)}    # Orange = ball
    CLASS_NAMES  = {0: 'player', 1: 'referee', 2: 'ball'}

    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    axes = axes.flatten()

    for idx, img_path in enumerate(sample_imgs):
        img = cv2.imread(img_path)
        if img is None:
            axes[idx].axis('off')
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        axes[idx].imshow(img)

        lbl_path = (img_path
                    .replace('/images/', '/labels/')
                    .replace('.jpg', '.txt')
                    .replace('.png', '.txt'))

        n_boxes = 0
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    color = CLASS_COLORS.get(cls_id, (1, 1, 1))
                    rect  = mpatches.Rectangle(
                        (x1, y1), bw*w, bh*h,
                        linewidth=2.5, edgecolor=color, facecolor='none')
                    axes[idx].add_patch(rect)
                    axes[idx].text(
                        x1, y1 - 4,
                        CLASS_NAMES.get(cls_id, '?'),
                        fontsize=8, color='white',
                        bbox=dict(boxstyle='round,pad=0.2',
                                  facecolor=color, alpha=0.9))
                    n_boxes += 1

        axes[idx].axis('off')
        axes[idx].set_title(f'{Path(img_path).name[:25]}  ({n_boxes} boxes)', fontsize=7)

    for idx in range(len(sample_imgs), 9):
        axes[idx].axis('off')

    # Legend
    handles = [
        mpatches.Patch(color=(0.2, 0.6, 1.0), label='Player'),
        mpatches.Patch(color=(1.0, 0.25, 0.25), label='Referee'),
        mpatches.Patch(color=(1.0, 0.6, 0.0), label='Ball'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=12,
               bbox_to_anchor=(0.5, -0.01))
    fig.suptitle('Merged Dataset — Annotated Training Samples\n'
                 'Verifying bounding boxes before training', fontsize=13)
    plt.tight_layout()
    plt.savefig('/content/sample_annotations.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: /content/sample_annotations.png')
    print(f'   Displayed {len(sample_imgs)} random training samples')


---
## 🚀 CELL 9 — Train YOLOv11m ⭐ (Main Training Step)

**What this does:**
Fine-tunes YOLOv11m on YOUR merged basketball dataset.

**Fine-tuning vs Training from scratch:**
| Approach | Weights start | Time needed | Data needed | Result |
|----------|--------------|------------|------------|--------|
| From scratch | Random | Days/weeks | Millions of images | Bad |
| **Fine-tuning ✅** | COCO pretrained | 60–90 min | 3600 images | Excellent |

YOLOv11m was pretrained on COCO (80 classes including "person"). It already knows what humans look like. We just teach it basketball-specific features.

**Key training settings explained:**
| Setting | Value | Why |
|---------|-------|-----|
| `lr0=0.001` | Low | Gentle fine-tuning (don't overwrite pretrained knowledge) |
| `mosaic=1.0` | ON | Combines 4 images per batch — great for crowded court scenes |
| `flipud=0.0` | OFF | Upside-down basketball court makes no sense |
| `patience=20` | 20 | Stops training early if no improvement for 20 epochs |

**Expected time:** ~60–90 minutes on T4 GPU

**Output:** `/content/runs/yolov11m_basketball/weights/best.pt`

**⚠️ Optional changes:**
- `epochs=100` → change to `150` for higher accuracy
- `batch=16` → change to `8` if you get `CUDA out of memory` error

In [ ]:
# ================================================================
# CELL 9 — TRAIN YOLOv11m (MAIN TRAINING STEP)
# ================================================================
# What: Fine-tunes YOLOv11m (pretrained on COCO) on our basketball data
# Why:  Fine-tuning = starts from human-detection weights, not random.
#       Needs only 3600 images and 90 minutes instead of days.
# Algorithm: YOLOv11m chosen over Faster-RCNN (too slow) and
#            DETR (needs too much data to converge).
# Where used: This is the DETECTION stage. ByteTrack (Cell 11) uses
#             these detections as input for the TRACKING stage.
#
# ⚠️  OPTIONAL CHANGES:
#   epochs = 100   →  change to 150 for higher accuracy (takes longer)
#   batch  = 16    →  change to 8 if you see "CUDA out of memory"
# ================================================================

from ultralytics import YOLO

# Load YOLOv11m pretrained on COCO — auto-downloads ~50MB
model = YOLO('yolo11m.pt')

print('🚀 Starting YOLOv11m training...')
print()
print('   Model     : YOLOv11m (pretrained COCO → fine-tuning)')
print('   Dataset   : /content/merged_dataset (~3600 images)')
print('   Classes   : player(0), referee(1), ball(2)')
print('   Device    : GPU (T4)')
print('   Est. time : 60–90 minutes')
print()

results = model.train(
    # ── Dataset ──────────────────────────────────────────────
    data     = '/content/merged_dataset/data.yaml',

    # ── Core Settings ────────────────────────────────────────
    epochs   = 100,    # ← CHANGE to 150 for more accuracy
    imgsz    = 640,    # YOLOv11 standard input size
    batch    = 16,     # ← CHANGE to 8 if CUDA out of memory
    device   = 0,      # GPU (0 = first available GPU)
    workers  = 2,

    # ── Optimizer ────────────────────────────────────────────
    # AdamW = better than SGD for fine-tuning pretrained models
    optimizer     = 'AdamW',
    lr0           = 0.001,    # low initial LR = gentle fine-tuning
    lrf           = 0.01,     # final LR = lr0 * lrf
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 5,        # slowly ramp LR up over first 5 epochs

    # ── Augmentation (simulates diverse conditions) ───────────
    # These prevent overfitting and improve generalization
    hsv_h      = 0.015,   # hue shift — handles different court lighting
    hsv_s      = 0.7,     # saturation jitter
    hsv_v      = 0.4,     # brightness jitter
    degrees    = 5.0,     # small rotation — cameras aren't always level
    translate  = 0.1,     # slight position shift
    scale      = 0.5,     # zoom in/out — players at different distances
    fliplr     = 0.5,     # horizontal flip — left/right court symmetry
    flipud     = 0.0,     # NO vertical flip — upside-down court is wrong
    mosaic     = 1.0,     # combine 4 images — excellent for crowd scenes
    mixup      = 0.15,    # blend 2 images together
    copy_paste = 0.1,     # copy players between images

    # ── Quality Settings ─────────────────────────────────────
    pretrained  = True,   # USE COCO pretrained weights (this IS fine-tuning)
    patience    = 20,     # stop if no improvement for 20 consecutive epochs
    conf        = 0.001,  # low conf during val = most accurate mAP calc
    iou         = 0.6,

    # ── Saving ───────────────────────────────────────────────
    save        = True,
    save_period = 10,     # checkpoint every 10 epochs
    plots       = True,   # auto-generate all loss/mAP charts
    val         = True,

    # ── Output Path ──────────────────────────────────────────
    project     = '/content/runs',
    name        = 'yolov11m_basketball',
    exist_ok    = True,
    verbose     = True,
)

# Save path for use in later cells
BEST_MODEL_PATH = '/content/runs/yolov11m_basketball/weights/best.pt'

print()
print('=' * 60)
print('  TRAINING COMPLETE!')
print('=' * 60)
print(f'  Best model saved : {BEST_MODEL_PATH}')
print(f'  Last checkpoint  : /content/runs/yolov11m_basketball/weights/last.pt')
print(f'  Training charts  : /content/runs/yolov11m_basketball/')
print('=' * 60)


---
## 📈 CELL 10 — Show Training Curves + Results

**What this does:**
Displays the auto-generated training charts:
- **results.png** — loss curves + mAP over 100 epochs
- **confusion_matrix.png** — which classes are confused with each other
- **PR_curve.png** — Precision vs Recall tradeoff
- **F1_curve.png** — F1 score vs confidence threshold

**How to read the training curves:**
- Loss should go DOWN over epochs (model is learning)
- mAP should go UP over epochs (model is getting better)
- If both are flat/zigzag → training is unstable (try lower lr0)

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 10 — DISPLAY TRAINING RESULTS AND CURVES
# ================================================================
# What: Shows loss curves, confusion matrix, PR curve, F1 curve
# Why:  Visual confirmation that training converged properly.
#       If loss didn't decrease → training failed → investigate.
# Change: Nothing. Just run.
# ================================================================

from IPython.display import Image, display

RUN_DIR = '/content/runs/yolov11m_basketball/'

print('📈 Training Results:')
print()

for plot_file, description in [
    ('results.png',          '📊 Loss Curves + mAP over Epochs'),
    ('confusion_matrix.png', '📊 Confusion Matrix (which classes get confused)'),
    ('PR_curve.png',         '📊 Precision-Recall Curve'),
    ('F1_curve.png',         '📊 F1 Score vs Confidence Threshold'),
    ('val_batch0_pred.jpg',  '📊 Sample Validation Predictions'),
]:
    full_path = RUN_DIR + plot_file
    if os.path.exists(full_path):
        print(f'{description}:')
        display(Image(filename=full_path, width=800))
        print()
    else:
        print(f'   (not found: {plot_file})')

print('✅ Training charts displayed!')


---
## 📊 CELL 11 — Evaluate Detection Accuracy (mAP, Precision, Recall, F1)

**What this does:**
Runs the trained model on the TEST set (images the model has never seen) and reports industry-standard detection metrics.

**Metrics explained:**
| Metric | Formula | What it means | Target |
|--------|---------|--------------|--------|
| **mAP@50** | mean AP at IoU=0.5 | Main accuracy metric | > 0.85 |
| **mAP@50-95** | mean AP across IoU 0.5–0.95 | Stricter accuracy | > 0.65 |
| **Precision** | TP/(TP+FP) | Of all detected players, how many are real? | > 0.85 |
| **Recall** | TP/(TP+FN) | Of all real players, how many did we find? | > 0.85 |
| **F1** | 2×P×R/(P+R) | Harmonic mean of precision and recall | > 0.85 |

**IoU (Intersection over Union):**
Measures how well the predicted box overlaps with the ground truth box. IoU=1.0 is perfect. IoU=0.5 means 50% overlap.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 11 — EVALUATE DETECTION ACCURACY
# ================================================================
# What: Runs trained model on TEST set, computes mAP/Precision/Recall
# Why:  Measures actual detection accuracy on unseen data.
#       mAP@50 is the primary metric (industry standard for detection).
# Target: mAP@50 > 0.85 = Good,  mAP@50 > 0.90 = Very Good
# Change: Nothing. Just run.
# ================================================================

from ultralytics import YOLO

best_model = YOLO(BEST_MODEL_PATH)

print('📊 Running evaluation on TEST set...')
print('   (Using images the model has never seen during training)')
print()

metrics = best_model.val(
    data      = '/content/merged_dataset/data.yaml',
    split     = 'test',
    conf      = 0.001,   # low conf = most complete mAP calculation
    iou       = 0.6,
    plots     = True,
    save_json = True,
    verbose   = False,
)

p  = metrics.box.mp
r  = metrics.box.mr
f1 = 2 * p * r / (p + r + 1e-9)

print()
print('=' * 60)
print('  DETECTION EVALUATION RESULTS (on TEST set)')
print('=' * 60)
print(f'  mAP@50       :  {metrics.box.map50:.4f}   ← target > 0.85')
print(f'  mAP@50-95    :  {metrics.box.map:.4f}   ← target > 0.65')
print(f'  Precision    :  {p:.4f}')
print(f'  Recall       :  {r:.4f}')
print(f'  F1 Score     :  {f1:.4f}')
print('-' * 60)
print('  Per-class mAP@50:')
class_names = ['player', 'referee', 'ball']
for i, name in enumerate(class_names):
    try:
        val = metrics.box.maps[i]
        bar = '█' * int(val * 20)
        print(f'    {name:10s}: {val:.4f}  {bar}')
    except Exception:
        pass
print('=' * 60)

# Save evaluation score for final report
EVAL_RESULTS = {
    'mAP50':     metrics.box.map50,
    'mAP50_95':  metrics.box.map,
    'precision': p,
    'recall':    r,
    'f1':        f1,
}
print()
print('✅ Evaluation complete!')


---
## 🎬 CELL 12 — Download Basketball Test Video

**What this does:**
Downloads a free, copyright-free basketball game video from Pexels for use as tracking input.

**Why Pexels?**
- 100% free, no login needed
- Direct MP4 download link (no scraping)
- High-quality HD footage

**⚠️ IF AUTO-DOWNLOAD FAILS (3 options):**

**Option A:** Upload manually
1. Go to https://www.pexels.com/search/videos/basketball/
2. Download any video to your PC
3. Colab left sidebar → Files icon → Upload
4. Change `INPUT_VIDEO = '/content/your_filename.mp4'` below

**Option B:** Use Google Drive
```python
from google.colab import drive
drive.mount('/content/drive')
INPUT_VIDEO = '/content/drive/MyDrive/your_basketball_video.mp4'
```

**Option C:** Use yt-dlp
```python
!pip install yt-dlp -q
!yt-dlp -f "best[height<=720]" "YOUTUBE_URL" -o "/content/game.mp4"
INPUT_VIDEO = '/content/game.mp4'
```

In [ ]:
# ================================================================
# CELL 12 — DOWNLOAD TEST VIDEO
# ================================================================
# What: Downloads a free basketball game video for tracking
# Why:  We need a real game video to test the full pipeline
# Source: Pexels (free, no copyright, direct MP4 link)
#
# ⚠️  IF AUTO-DOWNLOAD FAILS:
#   Option A: Go to pexels.com/search/videos/basketball/
#             Download any video, upload to Colab via Files panel
#             Then set: INPUT_VIDEO = '/content/your_filename.mp4'
#   Option B: Mount Drive and use a video from there
#   Option C: Use yt-dlp with a YouTube URL
# ================================================================

import urllib.request

# Free basketball videos from Pexels (no login needed)
VIDEO_URLS = [
    'https://videos.pexels.com/video-files/8425786/8425786-hd_1920_1080_25fps.mp4',
    'https://videos.pexels.com/video-files/6077784/6077784-hd_1280_720_25fps.mp4',
    'https://videos.pexels.com/video-files/5473157/5473157-hd_1280_720_24fps.mp4',
    'https://videos.pexels.com/video-files/3195394/3195394-hd_1280_720_25fps.mp4',
]

INPUT_VIDEO  = None
OUTPUT_VIDEO = '/content/basketball_tracked.mp4'

print('⬇️  Attempting to download basketball game video...')
print()

for url in VIDEO_URLS:
    save_path = '/content/basketball_game.mp4'
    try:
        print(f'   Trying: ...{url[-40:]}')
        urllib.request.urlretrieve(url, save_path)
        size_mb = os.path.getsize(save_path) / (1024 * 1024)
        if size_mb > 1.0:
            INPUT_VIDEO = save_path
            print(f'   ✅ Downloaded! Size: {size_mb:.1f} MB')
            break
        else:
            print(f'   ❌ Too small ({size_mb:.1f} MB), trying next...')
    except Exception as e:
        print(f'   ❌ Failed: {e}')

if INPUT_VIDEO is None:
    print()
    print('⚠️  All auto-downloads failed.')
    print('   Upload a video manually and update this line:')
    # ══════════════════════════════════════════════════
    # ⚠️  CHANGE THIS if you upload manually:
    INPUT_VIDEO = '/content/basketball_game.mp4'
    # ══════════════════════════════════════════════════
    print(f'   INPUT_VIDEO is set to: {INPUT_VIDEO}')
else:
    cap    = cv2.VideoCapture(INPUT_VIDEO)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print()
    print('📹 Video Info:')
    print(f'   File       : {INPUT_VIDEO}')
    print(f'   Resolution : {width}×{height}')
    print(f'   FPS        : {fps:.1f}')
    print(f'   Duration   : {frames/fps:.1f} seconds  ({frames} total frames)')
    print(f'   Output will be saved to: {OUTPUT_VIDEO}')


---
## 🎞️ CELL 13 — Extract + Preprocess Frames from Video

**What this does:**
1. Extracts frames at **5 FPS** (every 6th frame at 30fps video)
2. Skips **blurry frames** (blur score < 50) automatically
3. Shows 4 sample extracted frames for visual inspection

**Why extract frames?**
- Allows detection testing on individual frames before full video processing
- Provides visual confirmation the video is readable
- The blur filter ensures we only keep sharp, clear frames

**Why 5 FPS sampling?**
At 30fps basketball video, consecutive frames are nearly identical. Sampling at 5fps gives us diverse frames without redundancy.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 13 — EXTRACT + PREPROCESS FRAMES FROM VIDEO
# ================================================================
# What: Samples frames at 5 FPS, skips blurry frames
# Why:  (1) Visual check that video loaded correctly
#       (2) Provides frames for detection test (Cell 14)
#       (3) Blur filter = only sharp, usable frames
# Change: Nothing. Just run.
# ================================================================

os.makedirs('/content/extracted_frames', exist_ok=True)

if not os.path.exists(INPUT_VIDEO):
    print(f'❌ Video not found: {INPUT_VIDEO}')
    print('   Run Cell 12 first or set INPUT_VIDEO to your file path.')
else:
    cap          = cv2.VideoCapture(INPUT_VIDEO)
    video_fps    = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval     = max(1, int(video_fps / 5))  # extract 5 frames per second

    saved   = 0
    skipped_blur = 0

    print(f'🎞️  Extracting frames from: {INPUT_VIDEO}')
    print(f'   Video FPS     : {video_fps:.1f}')
    print(f'   Total frames  : {total_frames}')
    print(f'   Extract every : {interval} frames (= 5 FPS sampling)')
    print()

    for frame_idx in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % interval != 0:
            continue

        # Skip blurry frames
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur  = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur < 50:
            skipped_blur += 1
            continue

        out_path = f'/content/extracted_frames/frame_{saved:05d}.jpg'
        cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved += 1

    cap.release()

    print(f'   ✅ Extracted     : {saved} frames')
    print(f'   ⚠️  Skipped (blur): {skipped_blur} frames')
    print()

    # Show 4 sample frames
    frame_files = sorted(glob.glob('/content/extracted_frames/*.jpg'))[:4]
    if frame_files:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        for ax, fp in zip(axes, frame_files):
            img = cv2.imread(fp)
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis('off')
            ax.set_title(Path(fp).name, fontsize=8)
        plt.suptitle('Sample Extracted Frames (quality-filtered)', fontsize=12)
        plt.tight_layout()
        plt.savefig('/content/sample_frames.png', dpi=100)
        plt.show()
        print('✅ Sample frames displayed and saved!')


---
## 🔍 CELL 14 — Test Detection on Sample Frames

**What this does:**
Runs the trained YOLOv11m model on 6 extracted frames and draws bounding boxes. This is a visual quality check of detection accuracy before processing the full video.

**What to look for:**
- Players should have blue boxes with confidence scores
- Boxes should be tight around the players, not too large/small
- mAP@50 > 0.85 trained = you should see good boxes here

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 14 — TEST DETECTION ON SAMPLE FRAMES
# ================================================================
# What: Runs YOLOv11m on 6 frames to visually check detection quality
# Why:  Confidence check before committing to full video processing.
#       If boxes look bad here, check training (Cell 9) results.
# Where detection is used: YOLOv11m detects per frame.
#                          ByteTrack (Cell 15) then links those
#                          detections into consistent tracks.
# Change: Nothing. Just run.
# ================================================================

from ultralytics import YOLO

best_model  = YOLO(BEST_MODEL_PATH)
frame_files = sorted(glob.glob('/content/extracted_frames/*.jpg'))[:6]

if not frame_files:
    print('❌ No extracted frames. Run Cell 13 first.')
else:
    CLS_COLORS = {0: (0, 120, 255),   # Blue  = Player
                  1: (0,   0, 220),   # Red   = Referee
                  2: (0, 200, 255)}   # Cyan  = Ball
    CLS_NAMES  = {0: 'Player', 1: 'Referee', 2: 'Ball'}

    cols = 3
    rows = 2
    fig, axes = plt.subplots(rows, cols, figsize=(20, 12))
    axes = axes.flatten()

    for idx, frame_path in enumerate(frame_files):
        frame  = cv2.imread(frame_path)
        result = best_model.predict(frame, conf=0.35, iou=0.5, verbose=False)[0]

        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            color  = CLS_COLORS.get(cls_id, (255, 255, 255))
            label  = f'{CLS_NAMES.get(cls_id, "?")} {conf:.2f}'
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1-th-6), (x1+tw+4, y1), color, -1)
            cv2.putText(frame, label, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        axes[idx].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[idx].axis('off')
        axes[idx].set_title(
            f'Frame {idx+1}  |  {len(result.boxes)} detections', fontsize=10)

    for idx in range(len(frame_files), rows*cols):
        axes[idx].axis('off')

    plt.suptitle('Detection Test — YOLOv11m on Video Frames\n'
                 'Blue=Player  Red=Referee  Cyan=Ball', fontsize=13)
    plt.tight_layout()
    plt.savefig('/content/detection_test.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Detection test complete! Saved: /content/detection_test.png')


---
## 🏃 CELL 15 — Full Player Tracking with ByteTrack ⭐ (Main Tracking Step)

**What this does:**
Processes EVERY frame of the input video:
1. **YOLOv11m detects** all players, referees, and ball in each frame
2. **ByteTrack assigns** consistent ID numbers to each detected person
3. **Supervision draws** colored boxes, ID labels, and movement trails
4. **Saves** the fully annotated output video

---

### 🧠 WHY BYTETRACK IS USED HERE (and not SORT or DeepSORT)

**The core problem with SORT:**
SORT uses only a Kalman filter (motion prediction). When Player #3 crosses in front of Player #7, their Kalman-predicted positions overlap. SORT can't resolve this → assigns a new ID to one of them → tons of ID switches.

**The core problem with DeepSORT:**
DeepSORT adds a visual appearance model (CNN) to help re-identify players. But basketball players on the same team wear IDENTICAL jerseys. The appearance model sees two players that look exactly the same → confused → still swaps IDs.

**Why ByteTrack solves both problems:**
ByteTrack's innovation: instead of using only high-confidence detections for matching (like SORT/DeepSORT), it uses ALL detections in a **two-stage process**:
1. First pass: match tracks to high-confidence detections
2. Second pass: match remaining unmatched tracks to LOW-confidence detections

During occlusion (player partly hidden), the low-confidence detection is still there but SORT/DeepSORT throw it away. ByteTrack uses it to maintain the track → no ID switch.

**Result:** ByteTrack achieves MOTA=77.3%, MOTP=82.6%, only 558 ID switches at 171 FPS.

---

**What you'll see in the output video:**
- Colored bounding boxes around each player
- `#ID ClassName Confidence` label above each box
- 50-frame movement trail showing where each player came from
- HUD bar showing frame number and player count

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 15 — FULL PLAYER TRACKING WITH BYTETRACK
# ================================================================
# What: Processes every frame of the video with:
#       YOLOv11m (detection) + ByteTrack (tracking)
#
# WHY BYTETRACK (not SORT or DeepSORT):
#   SORT      → only Kalman filter → loses ID when players cross ❌
#   DeepSORT  → appearance CNN → confused by same team jerseys ❌
#   ByteTrack → uses ALL detections (high + low confidence) →
#               recovers tracks after occlusion → fewest ID switches ✅
#               171 FPS on T4 GPU ✅
#
# WHERE ALGORITHM IS USED:
#   model.track(..., tracker='bytetrack.yaml') — this single line
#   activates ByteTrack for every frame. The 'persist=True' flag
#   tells ByteTrack to maintain track IDs across frames.
#
# Change: Nothing. Just run.
# ================================================================

import supervision as sv
from ultralytics import YOLO

best_model  = YOLO(BEST_MODEL_PATH)
CLASS_NAMES = {0: 'Player', 1: 'Referee', 2: 'Ball'}

# Verify input video exists
if not os.path.exists(INPUT_VIDEO):
    print(f'❌ Input video not found: {INPUT_VIDEO}')
    print('   Run Cell 12 first.')
else:
    video_info = sv.VideoInfo.from_video_path(INPUT_VIDEO)
    print('📹 Video Details:')
    print(f'   Input      : {INPUT_VIDEO}')
    print(f'   Output     : {OUTPUT_VIDEO}')
    print(f'   Resolution : {video_info.width}×{video_info.height}')
    print(f'   FPS        : {video_info.fps}')
    print(f'   Frames     : {video_info.total_frames}')
    print(f'   Duration   : {video_info.total_frames/video_info.fps:.1f} seconds')
    print()
    print('🏃 Processing with ByteTrack...')
    print()

    # ── Annotators (from supervision library) ─────────────────
    box_annotator   = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(
        text_scale=0.5, text_thickness=1, text_padding=3)
    # TraceAnnotator draws the 50-frame movement trail behind each player
    trace_annotator = sv.TraceAnnotator(
        thickness=2, trace_length=50)

    # ── Tracking data collector (used for metrics in Cell 16) ──
    tracking_data = []

    # ── Main processing loop ──────────────────────────────────
    frame_gen = sv.get_video_frames_generator(INPUT_VIDEO)

    with sv.VideoSink(OUTPUT_VIDEO, video_info) as sink:
        for frame_idx, frame in enumerate(frame_gen):

            # ─── DETECTION + BYTETRACK TRACKING ──────────────
            # This is where ByteTrack is called.
            # model.track() runs YOLOv11m detection on the frame,
            # then passes the detections to ByteTrack which assigns
            # and maintains consistent track IDs across frames.
            # persist=True means ByteTrack remembers tracks between calls.
            result = best_model.track(
                frame,
                tracker = 'bytetrack.yaml',  # ← ByteTrack algorithm
                persist = True,    # keep IDs consistent across frames
                conf    = 0.35,    # confidence threshold
                iou     = 0.5,     # NMS IoU threshold
                verbose = False,
            )[0]
            # ─────────────────────────────────────────────────

            detections = sv.Detections.from_ultralytics(result)

            # Build text labels: #ID ClassName Confidence
            labels = []
            for i in range(len(detections)):
                tid  = (int(detections.tracker_id[i])
                        if detections.tracker_id is not None else -1)
                cls  = int(detections.class_id[i])
                conf = float(detections.confidence[i])
                name = CLASS_NAMES.get(cls, '?')
                labels.append(f'#{tid} {name} {conf:.2f}')

                # Collect data for metrics
                if detections.tracker_id is not None:
                    tracking_data.append({
                        'frame'    : frame_idx,
                        'track_id' : tid,
                        'class_id' : cls,
                        'conf'     : conf,
                        'xyxy'     : detections.xyxy[i].tolist(),
                    })

            # Draw trails, boxes, labels on the frame
            annotated = frame.copy()
            annotated = trace_annotator.annotate(annotated, detections)
            annotated = box_annotator.annotate(annotated, detections)
            annotated = label_annotator.annotate(annotated, detections, labels)

            # HUD bar at top of frame
            n_players  = sum(1 for c in detections.class_id if c == 0)
            n_referees = sum(1 for c in detections.class_id if c == 1)
            hud = (f'Frame:{frame_idx:4d} | '
                   f'Players:{n_players} | '
                   f'Refs:{n_referees} | '
                   f'Algorithm:ByteTrack+YOLOv11m')
            cv2.rectangle(annotated, (0, 0), (620, 32), (0, 0, 0), -1)
            cv2.putText(annotated, hud, (6, 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

            sink.write_frame(annotated)

            # Progress update every 60 frames
            if frame_idx % 60 == 0:
                pct = frame_idx / video_info.total_frames * 100
                print(f'   Frame {frame_idx:4d}/{video_info.total_frames} '
                      f'({pct:5.1f}%)  detections: {len(detections)}')

    print()
    print('=' * 60)
    print('  TRACKING COMPLETE!')
    print('=' * 60)
    print(f'  Output video : {OUTPUT_VIDEO}')
    print(f'  Data points  : {len(tracking_data)} tracked detections')
    print('=' * 60)


---
## 📊 CELL 16 — Tracking Statistics + Metrics

**What this does:**
Analyzes the tracking data collected during Cell 15 and computes:

| Metric | What it measures |
|--------|-----------------|
| **Unique track IDs** | Total distinct players identified |
| **Track length** | How many frames each player was continuously tracked |
| **Short tracks** | Tracks < 5 frames = likely ID switches |
| **Long tracks** | Tracks ≥ 20 frames = stable, consistent tracking |
| **Approx. MOTA** | Multi-Object Tracking Accuracy proxy score |

**Track length distribution chart:**
- Many long tracks (green zone) = ByteTrack is working well
- Many short tracks (red zone) = ID switches happening often

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 16 — TRACKING STATISTICS + METRICS
# ================================================================
# What: Computes tracking quality metrics from Cell 15 data
# Metrics:
#   - Track continuity (avg/max track length)
#   - ID switches (short tracks < 5 frames = bad)
#   - Approximate MOTA score
# Why:  Industry standard metrics for MOT evaluation.
#       IDF1/MOTA/MOTP (formal) require ground-truth annotations.
#       These proxy metrics give a good picture without GT.
# Change: Nothing. Just run.
# ================================================================

if 'tracking_data' not in dir() or not tracking_data:
    print('❌ No tracking data found. Run Cell 15 first.')
else:
    df = pd.DataFrame(tracking_data)
    player_df  = df[df['class_id'] == 0]
    referee_df = df[df['class_id'] == 1]
    ball_df    = df[df['class_id'] == 2]

    track_lens   = df.groupby('track_id').size()
    short_tracks = (track_lens < 5).sum()
    long_tracks  = (track_lens >= 20).sum()
    long_pct     = long_tracks / max(len(track_lens), 1) * 100
    mota_approx  = max(0.0, 1.0 - (short_tracks / max(len(df), 1)))

    print('=' * 60)
    print('  TRACKING STATISTICS — ByteTrack Results')
    print('=' * 60)
    print(f'  Total tracked detections   : {len(df):,}')
    print(f'  Unique track IDs           : {df["track_id"].nunique()}')
    print(f'  ── By class ──')
    print(f'  Player detections          : {len(player_df):,}')
    print(f'  Referee detections         : {len(referee_df):,}')
    print(f'  Ball detections            : {len(ball_df):,}')
    print(f'  ── Confidence ──')
    print(f'  Avg detection confidence   : {df["conf"].mean():.4f}')
    print(f'  Min detection confidence   : {df["conf"].min():.4f}')
    print(f'  ── Track Quality ──')
    print(f'  Avg track length           : {track_lens.mean():.1f} frames')
    print(f'  Max track length           : {track_lens.max()} frames')
    print(f'  Short tracks (< 5 frames)  : {short_tracks}  (fewer = better)')
    print(f'  Long tracks  (≥ 20 frames) : {long_pct:.1f}%  (higher = better)')
    print(f'  ── Score ──')
    print(f'  Approximate MOTA           : {mota_approx:.4f}')
    print('=' * 60)

    # Track length histogram
    plt.figure(figsize=(10, 4))
    plt.hist(track_lens.values, bins=30, color='#2196F3',
             edgecolor='black', linewidth=0.5)
    plt.axvline(x=5,  color='red',   linestyle='--', linewidth=2,
                label='< 5 frames = ID switch zone')
    plt.axvline(x=20, color='green', linestyle='--', linewidth=2,
                label='>= 20 frames = good stable track')
    plt.xlabel('Track Length (frames)', fontsize=12)
    plt.ylabel('Number of Tracks', fontsize=12)
    plt.title('ByteTrack — Track Length Distribution\n'
              'Long tracks = players tracked consistently', fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.savefig('/content/track_distribution.png', dpi=120)
    plt.show()
    print('✅ Track distribution chart saved!')


---
## 🔥 CELL 17 — Player Movement Heatmap (Bonus Marks)

**What this does:**
Generates a spatial heatmap showing WHERE each player spent time on the court during the tracked video.

**How to read it:**
- 🔴 **Red/Hot areas** = players spent the most time here
- 🔵 **Blue/Cool areas** = rarely visited
- Overlaid on the first frame of the video for court reference

**Why this is valuable:**
- Shows team positioning patterns (offense/defense zones)
- Identifies if a player was mostly in the paint vs perimeter
- Visual highlight for the project presentation

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 17 — PLAYER MOVEMENT HEATMAP (BONUS)
# ================================================================
# What: Generates spatial heatmap of player positions across the video
# Why:  Bonus visualization showing WHERE players spent time on court.
#       Red = most time, Blue = least time.
#       Overlaid on first frame for court reference.
# Change: Nothing. Just run.
# ================================================================

if 'tracking_data' not in dir() or not tracking_data:
    print('❌ No tracking data. Run Cell 15 first.')
else:
    cap = cv2.VideoCapture(INPUT_VIDEO)
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ret, bg_frame = cap.read()  # first frame as background
    cap.release()

    # Build heatmap by accumulating player center positions
    heatmap = np.zeros((H, W), dtype=np.float32)
    for item in tracking_data:
        if item['class_id'] != 0:  # players only
            continue
        x1, y1, x2, y2 = item['xyxy']
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        if 0 <= cy < H and 0 <= cx < W:
            heatmap[cy, cx] += 1

    # Smooth with Gaussian blur for visualization
    heatmap = cv2.GaussianBlur(heatmap, (61, 61), 0)
    if heatmap.max() > 0:
        heatmap = (heatmap / heatmap.max() * 255).astype(np.uint8)

    # Apply JET colormap (Blue=cold → Red=hot)
    colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Overlay heatmap on court background
    if bg_frame is not None:
        result_img = cv2.addWeighted(bg_frame, 0.35, colored, 0.65, 0)
    else:
        result_img = colored

    # Add colorbar legend
    cv2.putText(result_img, 'PLAYER MOVEMENT HEATMAP',
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
    cv2.putText(result_img, 'Red = Most time spent | Blue = Least time',
                (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)

    cv2.imwrite('/content/player_heatmap.jpg', result_img)

    # Display
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title('🔥 Player Movement Heatmap\n'
              'Red = Most time spent   |   Blue = Least time spent',
              fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('/content/heatmap_display.png', dpi=150, bbox_inches='tight')
    plt.show()

    total_players = len([d for d in tracking_data if d['class_id']==0])
    print(f'✅ Heatmap generated from {total_players:,} player position points')
    print(f'✅ Saved: /content/player_heatmap.jpg')


---
## 💾 CELL 18 — Auto-Save Everything to Google Drive

**What this does:**
Mounts Google Drive and saves ALL project outputs to a single organized folder: `Basketball_Project_Final/`

**Folder structure created:**
```
Basketball_Project_Final/
├── best_model_yolov11m.pt      ← trained model weights
├── basketball_tracked.mp4      ← output video with tracking
├── player_heatmap.jpg          ← movement heatmap
├── heatmap_display.png         ← heatmap display version
├── class_distribution.png      ← dataset class chart
├── sample_annotations.png      ← annotated training samples
├── detection_test.png          ← detection test on frames
├── track_distribution.png      ← tracking quality chart
├── data.yaml                   ← dataset config
└── training_run/               ← all training charts + checkpoints
    ├── results.png
    ├── confusion_matrix.png
    ├── PR_curve.png
    ├── F1_curve.png
    └── weights/
        ├── best.pt
        └── last.pt
```

**⚠️ Optional:** Change the folder name `Basketball_Project_Final` if you want.

After this runs, share the folder link as your submission.

In [ ]:
# ================================================================
# CELL 18 — AUTO-SAVE EVERYTHING TO GOOGLE DRIVE
# ================================================================
# What: Saves all outputs to a single organized Drive folder
# Why:  Colab loses all files when session disconnects.
#       Drive backup ensures nothing is lost.
#       Single folder = easy submission sharing.
#
# ⚠️  OPTIONAL CHANGE:
#   Rename 'Basketball_Project_Final' to any folder name you want
# ================================================================

from google.colab import drive

print('🔌 Mounting Google Drive...')
drive.mount('/content/drive', force_remount=False)

# ══════════════════════════════════════════════════════
# ⚠️  OPTIONAL: Change folder name if you want
SAVE_DIR = '/content/drive/MyDrive/Basketball_Project_Final/'
# ══════════════════════════════════════════════════════

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'📁 Saving to: {SAVE_DIR}')
print()

# Files to save: {source_path: destination_filename}
files_to_save = {
    BEST_MODEL_PATH                                           : 'best_model_yolov11m.pt',
    OUTPUT_VIDEO                                              : 'basketball_tracked.mp4',
    '/content/player_heatmap.jpg'                             : 'player_heatmap.jpg',
    '/content/heatmap_display.png'                            : 'heatmap_display.png',
    '/content/class_distribution.png'                         : 'class_distribution.png',
    '/content/sample_annotations.png'                         : 'sample_annotations.png',
    '/content/detection_test.png'                             : 'detection_test.png',
    '/content/track_distribution.png'                         : 'track_distribution.png',
    '/content/sample_frames.png'                              : 'sample_frames.png',
    '/content/merged_dataset/data.yaml'                       : 'data.yaml',
    '/content/runs/yolov11m_basketball/results.png'           : 'training_curves.png',
    '/content/runs/yolov11m_basketball/confusion_matrix.png'  : 'confusion_matrix.png',
    '/content/runs/yolov11m_basketball/PR_curve.png'          : 'PR_curve.png',
    '/content/runs/yolov11m_basketball/F1_curve.png'          : 'F1_curve.png',
}

saved   = []
missing = []
for src, dst_name in files_to_save.items():
    if os.path.exists(src):
        shutil.copy(src, SAVE_DIR + dst_name)
        saved.append(dst_name)
        print(f'   ✅ {dst_name}')
    else:
        missing.append(dst_name)
        print(f'   ⚠️  Not found (skipped): {dst_name}')

# Save full training run folder (all checkpoints + charts)
run_dir = '/content/runs/yolov11m_basketball/'
if os.path.exists(run_dir):
    shutil.copytree(run_dir, SAVE_DIR + 'training_run/', dirs_exist_ok=True)
    print(f'   ✅ training_run/ (full folder with all checkpoints)')

print()
print('=' * 60)
print(f'  SAVED {len(saved)} files to Google Drive')
print(f'  Folder: {SAVE_DIR}')
print('=' * 60)
print()
print('📤 To submit: Share this Drive folder with edit access')
print(f'   Drive → Right-click Basketball_Project_Final → Share')


---
## 🎨 CELL 19 — Live Demo with Gradio UI

**What this does:**
Launches an interactive web interface where you can:
- **Drag and drop** any video file from your PC, OR
- **Type a Google Drive path** directly

The tracked output video is shown in the same interface.

**How to use:**
1. Run this cell
2. A link appears: `Running on public URL: https://xxxxx.gradio.live`
3. Open that link in any browser
4. Upload a basketball video or paste a Drive path
5. Click "Track Video"
6. Watch the tracked video appear in the output panel

**This works without downloading anything — accessible from any device.**

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 19 — GRADIO LIVE DEMO
# ================================================================
# What: Launches interactive web UI for video tracking
# Why:  Lets evaluators/viewers try the system live without code.
#       Drag-and-drop input + video output in one browser window.
# How:  Gradio creates a public HTTPS URL that works anywhere.
#
# HOW TO USE:
#   1. Run this cell
#   2. Click the public URL that appears
#   3. Upload a basketball video (drag-drop or Drive path)
#   4. Click "Track Video"
#   5. Tracked output video appears in the Output section
#
# Change: Nothing. Just run.
# ================================================================

import gradio as gr
import tempfile
import os
from ultralytics import YOLO
import supervision as sv
import cv2
import numpy as np

# Load the trained model
print('Loading trained model for Gradio demo...')
demo_model = YOLO(BEST_MODEL_PATH)
print(f'✅ Model loaded: {BEST_MODEL_PATH}')
print()

def track_video(video_file, drive_path, progress=gr.Progress()):
    """
    Main tracking function called by the Gradio interface.
    Accepts either an uploaded file or a Google Drive path.
    Returns the path to the tracked output video.
    """
    # Determine input source
    if drive_path and drive_path.strip():
        input_path = drive_path.strip()
        if not os.path.exists(input_path):
            return None, f'❌ Drive path not found: {input_path}'
    elif video_file is not None:
        input_path = video_file
    else:
        return None, '❌ Please upload a video or enter a Drive path'

    try:
        # Setup output path
        with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
            out_path = tmp.name

        # Get video info
        video_info = sv.VideoInfo.from_video_path(input_path)
        total_frames = video_info.total_frames

        # Setup annotators
        box_annotator   = sv.BoxAnnotator(thickness=2)
        label_annotator = sv.LabelAnnotator(text_scale=0.5, text_padding=3)
        trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=40)

        CLASS_NAMES = {0: 'Player', 1: 'Referee', 2: 'Ball'}

        # Process video
        frame_gen = sv.get_video_frames_generator(input_path)
        player_counts = []

        with sv.VideoSink(out_path, video_info) as sink:
            for frame_idx, frame in enumerate(frame_gen):
                # ByteTrack tracking
                result = demo_model.track(
                    frame,
                    tracker  = 'bytetrack.yaml',
                    persist  = True,
                    conf     = 0.35,
                    iou      = 0.5,
                    verbose  = False,
                )[0]

                detections = sv.Detections.from_ultralytics(result)

                labels = []
                for i in range(len(detections)):
                    tid  = (int(detections.tracker_id[i])
                            if detections.tracker_id is not None else -1)
                    cls  = int(detections.class_id[i])
                    conf = float(detections.confidence[i])
                    name = CLASS_NAMES.get(cls, '?')
                    labels.append(f'#{tid} {name} {conf:.2f}')

                annotated = frame.copy()
                annotated = trace_annotator.annotate(annotated, detections)
                annotated = box_annotator.annotate(annotated, detections)
                annotated = label_annotator.annotate(annotated, detections, labels)

                n_players = sum(1 for c in detections.class_id if c == 0)
                player_counts.append(n_players)

                hud = f'Frame:{frame_idx:4d} | Players:{n_players} | ByteTrack+YOLOv11m'
                cv2.rectangle(annotated, (0, 0), (520, 32), (0, 0, 0), -1)
                cv2.putText(annotated, hud, (6, 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

                sink.write_frame(annotated)
                progress(frame_idx / max(total_frames, 1))

        avg_players = np.mean(player_counts) if player_counts else 0
        status_msg = (
            f'✅ Tracking complete!\n'
            f'Frames processed: {total_frames}\n'
            f'Avg players/frame: {avg_players:.1f}\n'
            f'Algorithm: ByteTrack + YOLOv11m'
        )
        return out_path, status_msg

    except Exception as e:
        return None, f'❌ Error: {str(e)}'


# ── Build Gradio Interface ────────────────────────────────────
with gr.Blocks(title='Basketball Tracking Demo', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏀 Basketball Player Detection & Tracking
    ## YOLOv11m + ByteTrack — Live Demo

    Upload a basketball video **or** paste a Google Drive path.
    The tracked output (with player IDs and movement trails) will appear below.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 📁 Input')

            video_input = gr.Video(
                label='Drag & Drop Video Here (or click to browse)',
                sources=['upload'],
            )

            gr.Markdown('**— OR —**')

            drive_input = gr.Textbox(
                label='Google Drive Path',
                placeholder='/content/drive/MyDrive/your_basketball_video.mp4',
                lines=1,
            )

            track_btn = gr.Button(
                '🏃 Track Video',
                variant='primary',
                size='lg',
            )

            status_box = gr.Textbox(
                label='Status',
                lines=5,
                interactive=False,
            )

        with gr.Column(scale=1):
            gr.Markdown('### 📹 Tracked Output')
            video_output = gr.Video(
                label='Tracked Video (players labeled with IDs + trails)',
                autoplay=True,
            )

    gr.Markdown("""
    ---
    **Algorithm:** ByteTrack (tracking) + YOLOv11m (detection)  
    **Classes detected:** Player 🔵 | Referee 🔴 | Ball 🟡  
    **Features:** Persistent player IDs across frames, 50-frame movement trails
    """)

    track_btn.click(
        fn      = track_video,
        inputs  = [video_input, drive_input],
        outputs = [video_output, status_box],
    )

print('🎨 Launching Gradio interface...')
print('   A public URL will appear below.')
print('   Open it in any browser on any device.')
print()
demo.launch(share=True, debug=False)


---
## 📋 CELL 20 — Final Project Summary Report

**What this does:**
Prints a complete summary of everything — model performance, tracking stats, algorithm choices, file paths, and submission instructions.

Copy this output into your project report / documentation.

**⚠️ Nothing to change. Just run it.**

In [ ]:
# ================================================================
# CELL 20 — FINAL SUMMARY REPORT
# ================================================================
# What: Prints complete project summary for documentation
# Why:  Single-view overview of everything done.
#       Copy-paste into your submission report.
# Change: Nothing. Just run.
# ================================================================

print('=' * 65)
print('  🏀 BASKETBALL PLAYER DETECTION & TRACKING — FINAL REPORT')
print('=' * 65)
print()
print('── ALGORITHM CHOICES & JUSTIFICATION ─────────────────────')
print()
print('  DETECTION:  YOLOv11m')
print('  Reason:     Single-stage detector. Real-time (30-100 FPS).')
print('              22% fewer params than YOLOv8 with higher mAP.')
print('              COCO pretrained = already knows human body shapes.')
print('              Faster R-CNN too slow (5-10 FPS). DETR needs')
print('              too much data to converge. YOLOv11m is optimal.')
print()
print('  TRACKING:   ByteTrack')
print('  Reason:     SORT loses IDs when players cross (Kalman only).')
print('              DeepSORT confused by same-color jerseys.')
print('              ByteTrack uses ALL detections (high + low conf)')
print('              = recovers tracks during occlusion = fewest ID')
print('              switches. MOTA=77.3%, 171 FPS. Best choice.')
print()
print('── WHERE ALGORITHMS ARE USED IN CODE ─────────────────────')
print()
print('  Detection:  Cell 9  — model.train() fine-tunes YOLOv11m')
print('              Cell 14 — model.predict() tests on sample frames')
print('              Cell 15 — model.track() runs detection per frame')
print()
print('  Tracking:   Cell 15 — model.track(tracker="bytetrack.yaml")')
print('                        ByteTrack links detections across frames')
print('                        persist=True maintains IDs between calls')
print()
print('── DATASETS ───────────────────────────────────────────────')
print()
print('  Dataset 1: basketball-player-detection (1616 images)')
print('  Dataset 2: basketball-player-detection-2 (1398 images)')
print('  Dataset 3: basketball-players-fy4c2 (600+ images)')
print('  Merged   : ~3600+ images total')
print('  Classes  : player(0), referee(1), ball(2)')
print()
print('── TRAINING SETUP ─────────────────────────────────────────')
print()
print('  Base model  : YOLOv11m (COCO pretrained)')
print('  Approach    : Fine-tuning (NOT training from scratch)')
print('  Epochs      : 100')
print('  Image size  : 640×640')
print('  Optimizer   : AdamW  lr0=0.001')
print('  Augments    : Mosaic, Mixup, CopyPaste, HSV, Flip')
print('  Hardware    : T4 GPU (Colab)')
print()

if 'EVAL_RESULTS' in dir():
    print('── DETECTION RESULTS ──────────────────────────────────────')
    print()
    print(f'  mAP@50       : {EVAL_RESULTS["mAP50"]:.4f}')
    print(f'  mAP@50-95    : {EVAL_RESULTS["mAP50_95"]:.4f}')
    print(f'  Precision    : {EVAL_RESULTS["precision"]:.4f}')
    print(f'  Recall       : {EVAL_RESULTS["recall"]:.4f}')
    print(f'  F1 Score     : {EVAL_RESULTS["f1"]:.4f}')
    print()

if 'tracking_data' in dir() and tracking_data:
    df_s = pd.DataFrame(tracking_data)
    tl   = df_s.groupby('track_id').size()
    print('── TRACKING RESULTS ───────────────────────────────────────')
    print()
    print(f'  Unique players tracked : {df_s["track_id"].nunique()}')
    print(f'  Total detections       : {len(df_s):,}')
    print(f'  Avg confidence         : {df_s["conf"].mean():.4f}')
    print(f'  Avg track length       : {tl.mean():.1f} frames')
    print(f'  Short tracks (<5f)     : {(tl<5).sum()}  (ID switch indicator)')
    print()

print('── OUTPUT FILES ───────────────────────────────────────────')
print()
print(f'  Trained model     : {BEST_MODEL_PATH}')
print(f'  Tracked video     : {OUTPUT_VIDEO}')
print(f'  Player heatmap    : /content/player_heatmap.jpg')
print(f'  Training charts   : /content/runs/yolov11m_basketball/')
if 'SAVE_DIR' in dir():
    print(f'  Google Drive      : {SAVE_DIR}')
print()
print('── SUBMISSION CHECKLIST ───────────────────────────────────')
print()
print('  ✅ Dataset downloaded + preprocessed (3 sources merged)')
print('  ✅ Quality filter applied (blur/dark/corrupted removed)')
print('  ✅ YOLOv11m fine-tuned on merged basketball dataset')
print('  ✅ Detection evaluated (mAP, Precision, Recall, F1)')
print('  ✅ ByteTrack tracking on full game video')
print('  ✅ Tracking metrics computed (MOTA proxy, track continuity)')
print('  ✅ Player movement heatmap (bonus)')
print('  ✅ Gradio live demo UI (bonus)')
print('  ✅ All files saved to Google Drive')
print()
print('=' * 65)
print('  Submission: Share the Colab notebook + Drive folder link')
print('=' * 65)
